In [13]:
import pandas as pd
import numpy as np
import scipy.io as sio
import torch
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from data_classes.decomposition import PCA_features

In [14]:
phase1_data = sio.loadmat('../data/mine_impact_data_2019.mat')
samples  = pd.DataFrame(phase1_data["x"].T)
labels  = pd.DataFrame(phase1_data["y"].T, columns=["y"])

df = pd.concat([samples, labels], axis=1, join="inner")

df = df.dropna()

shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_X = shuffled_df.iloc[:, :-1]
df_Y = shuffled_df.iloc[:, -1]

data_P1 = PCA_features(df_X, df_Y,explained_variance=0.95)

In [4]:
print(data_P1.get_samples().shape)

(3309, 396)


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, adjusted_rand_score


# X = df_X.values
# y = df_Y.values

X = data_P1.get_samples()
y = data_P1.get_labels()


X_train, X_test = X[:3000], X[3000:]
y_train, y_test = y[:3000], y[3000:]


kmeans = KMeans(n_clusters=2, random_state=42,init="k-means++")
kmeans.fit(X_train)   # no `y=`


preds = kmeans.predict(X_test)

mapping = {}
for cluster in np.unique(kmeans.labels_):
    labels_in_cluster = y_train[kmeans.labels_ == cluster]
    mapping[cluster] = np.bincount(labels_in_cluster).argmax()

#Convert cluster indices to class predictions
mapped_preds = np.vectorize(mapping.get)(preds)

#Evaluate
print("Clustering accuracy:", accuracy_score(y_test, mapped_preds))
print("Inertia (sum of squared distances):", kmeans.inertia_)

Clustering accuracy: 0.6957928802588996
Adjusted Rand Index: -0.003634317491854419
Inertia (sum of squared distances): 94903113.52705844


In [11]:
import plotly.express as px
from sklearn.decomposition import PCA


# ─────────── Plot 1: True Labels ───────────
fig_true = px.scatter_3d(
    x=X_test[:, 0], 
    y=X_test[:, 1], 
    z=y_test,
    color=y_test.astype(str),
    labels={'x':'PC1','y':'PC2',"z": "y_test",'color':'True Label'},
    title='True Labels in 2D PCA Space'
)
fig_true.show()

# ─────────── Plot 2: KMeans Clusters ───────────
fig_cluster = px.scatter(
    x=X_test[:, 0], 
    y=X_test[:, 1], 
    color=preds.astype(str),
    labels={'x':'PC1','y':'PC2','color':'Cluster'},
    title='KMeans Cluster Assignments in 2D PCA Space'
)
fig_cluster.show()